In [1]:
## Enhanced Range-based retracements during NY session (9:30am-4pm)
## Added: Open position tracking, distance buckets, percentage ranges, day-of-week analysis

import pandas as pd
from collections import defaultdict
import numpy as np

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data from 2013 to 2025
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

# Adjust the index to group by NY trading day (18:00 to 17:00)
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

# Define ranges to track (hour, minute) tuples for start and end
# Format: (start_hour, start_minute, end_hour, end_minute, is_previous_day)
RANGES_TO_TEST = {
    # Previous day ranges
    'Globex_6pm-7pm': (18, 0, 19, 0, True),
    'Globex_6pm-8pm': (18, 0, 20, 0, True),
    'Asia_8pm-9pm': (20, 0, 21, 0, True),
    'Asia_8pm-10pm': (20, 0, 22, 0, True),
    'Asia_10pm-11pm': (22, 0, 23, 0, True),
    'Asia_11pm-12am': (23, 0, 0, 0, True),  # Crosses midnight
    
    # Current day ranges - early morning
    'Midnight_12am-1am': (0, 0, 1, 0, False),
    'Midnight_12am-12:30am': (0, 0, 0, 30, False),
    'Late_Night_1am-2am': (1, 0, 2, 0, False),
    'London_2am-3am': (2, 0, 3, 0, False),
    'London_2am-2:30am': (2, 0, 2, 30, False),
    'London_2:30am-3am': (2, 30, 3, 0, False),
    'London_3am-4am': (3, 0, 4, 0, False),
    'London_4am-5am': (4, 0, 5, 0, False),
    'London_5am-6am': (5, 0, 6, 0, False),
    
    # Pre-market ranges
    'PreMarket_6am-7am': (6, 0, 7, 0, False),
    'PreMarket_7am-8am': (7, 0, 8, 0, False),
    'PreMarket_8am-9am': (8, 0, 9, 0, False),
    'PreMarket_8am-9:30am': (8, 0, 9, 30, False),
    'PreMarket_9am-9:30am': (9, 0, 9, 30, False),
    
    # 30-minute ranges
    'PreMarket_6am-6:30am': (6, 0, 6, 30, False),
    'PreMarket_6:30am-7am': (6, 30, 7, 0, False),
    'PreMarket_7am-7:30am': (7, 0, 7, 30, False),
    'PreMarket_7:30am-8am': (7, 30, 8, 0, False),
    'PreMarket_8am-8:30am': (8, 0, 8, 30, False),
    'PreMarket_8:30am-9am': (8, 30, 9, 0, False),
    'PreMarket_9am-9:30am': (9, 0, 9, 30, False),
}

# Distance buckets (in percentage from range midpoint)
DISTANCE_BUCKETS = [0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, float('inf')]

# Initialize tracking structures
retracement_stats = {
    'overall': {},
    'london_engulfs_asia': {},
    'asia_engulfs_london': {},
    'london_partially_engulfs_upwards': {},
    'london_partially_engulfs_downwards': {}
}

# Initialize each range
for pattern in retracement_stats.keys():
    for range_name in RANGES_TO_TEST.keys():
        retracement_stats[pattern][range_name] = {
            'total_days': 0,
            'retraced_days': 0,
            'time_slots': defaultdict(int),
            'high_taken_first_count': 0,
            'low_taken_first_count': 0,
            'avg_range_size': [],
            'avg_range_size_pct': [],
            
            # NEW: Open position tracking
            'opened_above_range': 0,
            'opened_below_range': 0,
            'opened_inside_range': 0,
            'retraced_when_above': 0,
            'retraced_when_below': 0,
            'retraced_when_inside': 0,
            
            # NEW: Distance buckets
            'distance_buckets': {i: {'total': 0, 'retraced': 0} for i in range(len(DISTANCE_BUCKETS))},
            
            # NEW: Day of week tracking
            'day_of_week': {
                i: {'total': 0, 'retraced': 0} for i in range(5)  # 0=Monday, 4=Friday
            }
        }

print("Processing range retracement data with enhanced tracking...")
processed_count = 0

for date in dates[1:]:  # Start from second day
    processed_count += 1
    if processed_count % 500 == 0:
        print(f"Processed {processed_count} days...")
    
    # Map the NY trading day back to the actual date
    actual_date = date + pd.Timedelta(hours=7)
    
    # Get day of week (0=Monday, 6=Sunday)
    day_of_week = actual_date.weekday()
    
    # Skip weekends
    if day_of_week >= 5:
        continue
    
    # Define session time ranges for pattern detection
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    london_start = asian_end
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    
    # Extract session data for pattern detection
    asian_data = df.loc[asian_start:asian_end]
    london_data = df.loc[london_start:london_end]
    
    # Skip if any session data is missing
    if asian_data.empty or london_data.empty:
        continue
    
    # Calculate session highs and lows for pattern detection
    asian_high = asian_data['High'].max()
    asian_low = asian_data['Low'].min()
    london_high = london_data['High'].max()
    london_low = london_data['Low'].min()
    
    # Skip if any high/low is NaN
    if pd.isna([asian_high, asian_low, london_high, london_low]).any():
        continue
    
    # Determine pattern
    current_pattern = None
    if london_high > asian_high and london_low < asian_low:
        current_pattern = 'london_engulfs_asia'
    elif london_high <= asian_high and london_low >= asian_low:
        current_pattern = 'asia_engulfs_london'
    elif london_high > asian_high and london_low >= asian_low:
        current_pattern = 'london_partially_engulfs_upwards'
    elif london_low < asian_low and london_high <= asian_high:
        current_pattern = 'london_partially_engulfs_downwards'
    
    # Get 9:30am open price
    open_930_time = actual_date.replace(hour=9, minute=30, second=0)
    open_930_data = df.loc[open_930_time:open_930_time]
    
    if open_930_data.empty:
        continue
    
    open_930_price = open_930_data.iloc[0]['Open']
    
    # Get the 9:30am-4pm session data for retracement checks
    retracement_session_start = actual_date.replace(hour=9, minute=30, second=0)
    retracement_session_end = actual_date.replace(hour=16, minute=0, second=0)
    retracement_session_data = df.loc[retracement_session_start:retracement_session_end]
    
    if retracement_session_data.empty:
        continue
    
    # Process each range
    for range_name, (start_hour, start_min, end_hour, end_min, is_prev_day) in RANGES_TO_TEST.items():
        # Determine the actual start and end times
        if is_prev_day:
            range_start = actual_date.replace(hour=start_hour, minute=start_min, second=0) - pd.Timedelta(days=1)
        else:
            range_start = actual_date.replace(hour=start_hour, minute=start_min, second=0)
        
        # Handle ranges that cross midnight
        if is_prev_day and end_hour == 0:
            range_end = actual_date.replace(hour=end_hour, minute=end_min, second=0)
        elif is_prev_day:
            range_end = actual_date.replace(hour=end_hour, minute=end_min, second=0) - pd.Timedelta(days=1)
        else:
            range_end = actual_date.replace(hour=end_hour, minute=end_min, second=0)
        
        # Get the range data
        range_data = df.loc[range_start:range_end]
        
        if range_data.empty:
            continue
        
        # Calculate range high, low, and 50% retracement
        range_high = range_data['High'].max()
        range_low = range_data['Low'].min()
        
        if pd.isna(range_high) or pd.isna(range_low):
            continue
        
        range_size = range_high - range_low
        range_midpoint = (range_high + range_low) / 2
        
        # Skip very small ranges (likely data issues or non-trading periods)
        if range_size < 1:  # Adjust threshold as needed
            continue
        
        # Calculate range size as percentage
        range_size_pct = (range_size / range_midpoint) * 100
        
        # Calculate 50% retracement level
        retracement_50 = range_low + (range_size * 0.5)
        
        # Determine where 9:30am open is relative to range
        if open_930_price > range_high:
            open_position = 'above'
        elif open_930_price < range_low:
            open_position = 'below'
        else:
            open_position = 'inside'
        
        # Calculate distance from range midpoint as percentage
        distance_from_mid = abs(open_930_price - range_midpoint)
        distance_pct = (distance_from_mid / range_midpoint) * 100
        
        # Find which bucket this distance falls into
        bucket_idx = None
        for i in range(len(DISTANCE_BUCKETS) - 1):
            if distance_pct >= DISTANCE_BUCKETS[i] and distance_pct < DISTANCE_BUCKETS[i+1]:
                bucket_idx = i
                break
        
        if bucket_idx is None:
            bucket_idx = len(DISTANCE_BUCKETS) - 2  # Last bucket
        
        # Check if 50% level was hit during 9:30am-4pm session
        was_retraced = False
        retracement_time = None
        high_taken_first = False
        low_taken_first = False
        
        for idx, row in retracement_session_data.iterrows():
            # Check if 50% level was touched
            if row['Low'] <= retracement_50 <= row['High']:
                if not was_retraced:
                    was_retraced = True
                    retracement_time = idx
                    break
            
            # Track if high was taken before 50%
            if not was_retraced and row['High'] >= range_high:
                high_taken_first = True
            
            # Track if low was taken before 50%
            if not was_retraced and row['Low'] <= range_low:
                low_taken_first = True
        
        # Helper function to update stats
        def update_stats(stats_dict):
            stats_dict['total_days'] += 1
            stats_dict['avg_range_size'].append(range_size)
            stats_dict['avg_range_size_pct'].append(range_size_pct)
            
            # Update open position tracking
            if open_position == 'above':
                stats_dict['opened_above_range'] += 1
                if was_retraced:
                    stats_dict['retraced_when_above'] += 1
            elif open_position == 'below':
                stats_dict['opened_below_range'] += 1
                if was_retraced:
                    stats_dict['retraced_when_below'] += 1
            else:
                stats_dict['opened_inside_range'] += 1
                if was_retraced:
                    stats_dict['retraced_when_inside'] += 1
            
            # Update distance bucket
            stats_dict['distance_buckets'][bucket_idx]['total'] += 1
            if was_retraced:
                stats_dict['distance_buckets'][bucket_idx]['retraced'] += 1
            
            # Update day of week
            stats_dict['day_of_week'][day_of_week]['total'] += 1
            if was_retraced:
                stats_dict['day_of_week'][day_of_week]['retraced'] += 1
            
            if was_retraced:
                stats_dict['retraced_days'] += 1
                
                # Track timing
                hour = retracement_time.hour
                minute = retracement_time.minute
                slot_minute = (minute // 15) * 15
                time_slot = f"{hour:02d}:{slot_minute:02d}"
                stats_dict['time_slots'][time_slot] += 1
                
                # Track what happened first
                if high_taken_first:
                    stats_dict['high_taken_first_count'] += 1
                if low_taken_first:
                    stats_dict['low_taken_first_count'] += 1
        
        # Update overall stats
        update_stats(retracement_stats['overall'][range_name])
        
        # Update pattern-specific stats if we have a pattern
        if current_pattern:
            update_stats(retracement_stats[current_pattern][range_name])

# ========================================================================
# DISPLAY RESULTS
# ========================================================================

def display_retracement_stats(stats, title, sort_by_probability=True, show_all=True):
    """Display retracement statistics with enhanced metrics"""
    print("\n" + "=" * 160)
    print(f"{title}")
    print("=" * 160)
    
    # Create list of ranges with their stats for sorting
    range_list = []
    for range_name, data in stats.items():
        if data['total_days'] == 0:
            continue
        
        retracement_pct = (data['retraced_days'] / data['total_days']) * 100
        avg_range = sum(data['avg_range_size']) / len(data['avg_range_size']) if data['avg_range_size'] else 0
        avg_range_pct = sum(data['avg_range_size_pct']) / len(data['avg_range_size_pct']) if data['avg_range_size_pct'] else 0
        
        range_list.append({
            'name': range_name,
            'data': data,
            'pct': retracement_pct,
            'avg_range': avg_range,
            'avg_range_pct': avg_range_pct
        })
    
    # Sort by retracement percentage (descending)
    if sort_by_probability:
        range_list.sort(key=lambda x: x['pct'], reverse=True)
    
    # Display header
    print(f"\n{'Rank'.ljust(6)} {'Range'.ljust(30)} {'Sample'.ljust(10)} {'Success %'.ljust(12)} {'Avg Range %'.ljust(15)} {'Open Above'.ljust(12)} {'Open Below'.ljust(12)} {'Open Inside'.ljust(12)}")
    print("-" * 160)
    
    display_count = len(range_list) if show_all else 20
    
    for rank, item in enumerate(range_list[:display_count], 1):
        range_name = item['name']
        data = item['data']
        retracement_pct = item['pct']
        avg_range_pct = item['avg_range_pct']
        
        # Calculate open position percentages
        total = data['total_days']
        above_pct = (data['opened_above_range'] / total * 100) if total > 0 else 0
        below_pct = (data['opened_below_range'] / total * 100) if total > 0 else 0
        inside_pct = (data['opened_inside_range'] / total * 100) if total > 0 else 0
        
        # Color coding for success rate
        success_indicator = "★★★" if retracement_pct >= 70 else "★★" if retracement_pct >= 60 else "★" if retracement_pct >= 50 else ""
        
        print(f"{str(rank).ljust(6)} {range_name.ljust(30)} {str(data['total_days']).ljust(10)} "
              f"{retracement_pct:6.2f}% {success_indicator.ljust(5)} {avg_range_pct:6.3f}%        "
              f"{above_pct:5.1f}%       {below_pct:5.1f}%       {inside_pct:5.1f}%")
    
    # Show detailed stats for top ranges
    print(f"\n{'='*160}")
    print(f"DETAILED STATS FOR TOP {min(10, len(range_list))} RANGES")
    print(f"{'='*160}")
    
    for rank, item in enumerate(range_list[:10], 1):
        range_name = item['name']
        data = item['data']
        retracement_pct = item['pct']
        avg_range = item['avg_range']
        avg_range_pct = item['avg_range_pct']
        
        print(f"\n{'-' * 160}")
        print(f"#{rank}: {range_name}")
        print(f"{'-' * 160}")
        print(f"Total days analyzed: {data['total_days']}")
        print(f"Days with 50% retracement: {data['retraced_days']} ({retracement_pct:.2f}%)")
        print(f"Average range size: {avg_range:.2f} points ({avg_range_pct:.3f}%)")
        
        # Open position analysis
        print(f"\n9:30AM OPEN POSITION RELATIVE TO RANGE:")
        if data['opened_above_range'] > 0:
            above_retrace_pct = (data['retraced_when_above'] / data['opened_above_range']) * 100
            print(f"  Above range: {data['opened_above_range']} times ({data['opened_above_range']/data['total_days']*100:.1f}%) → Retraced: {above_retrace_pct:.1f}%")
        if data['opened_below_range'] > 0:
            below_retrace_pct = (data['retraced_when_below'] / data['opened_below_range']) * 100
            print(f"  Below range: {data['opened_below_range']} times ({data['opened_below_range']/data['total_days']*100:.1f}%) → Retraced: {below_retrace_pct:.1f}%")
        if data['opened_inside_range'] > 0:
            inside_retrace_pct = (data['retraced_when_inside'] / data['opened_inside_range']) * 100
            print(f"  Inside range: {data['opened_inside_range']} times ({data['opened_inside_range']/data['total_days']*100:.1f}%) → Retraced: {inside_retrace_pct:.1f}%")
        
        # Distance bucket analysis
        print(f"\nRETRACEMENT BY DISTANCE FROM RANGE MIDPOINT:")
        print(f"{'Distance %'.ljust(20)} | {'Count'.ljust(10)} | {'Retracement %'.ljust(20)} | Bar Chart")
        print("-" * 100)
        
        for i in range(len(DISTANCE_BUCKETS) - 1):
            bucket_data = data['distance_buckets'][i]
            if bucket_data['total'] > 0:
                bucket_retrace_pct = (bucket_data['retraced'] / bucket_data['total']) * 100
                bar_length = int(bucket_retrace_pct / 2)
                bar = '█' * bar_length
                
                if DISTANCE_BUCKETS[i+1] == float('inf'):
                    bucket_label = f"{DISTANCE_BUCKETS[i]:.2f}%+"
                else:
                    bucket_label = f"{DISTANCE_BUCKETS[i]:.2f}%-{DISTANCE_BUCKETS[i+1]:.2f}%"
                
                print(f"{bucket_label.ljust(20)} | {str(bucket_data['total']).ljust(10)} | "
                      f"{bucket_retrace_pct:5.1f}% ({bucket_data['retraced']}/{bucket_data['total']}) | {bar}")
        
        # Day of week analysis
        print(f"\nRETRACEMENT BY DAY OF WEEK:")
        print(f"{'Day'.ljust(15)} | {'Count'.ljust(10)} | {'Retracement %'.ljust(20)} | Bar Chart")
        print("-" * 100)
        
        days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
        for i, day_name in enumerate(days):
            day_data = data['day_of_week'][i]
            if day_data['total'] > 0:
                day_retrace_pct = (day_data['retraced'] / day_data['total']) * 100
                bar_length = int(day_retrace_pct / 2)
                bar = '█' * bar_length
                
                print(f"{day_name.ljust(15)} | {str(day_data['total']).ljust(10)} | "
                      f"{day_retrace_pct:5.1f}% ({day_data['retraced']}/{day_data['total']}) | {bar}")
        
        if data['retraced_days'] > 0:
            # Show what happened first
            high_first_pct = (data['high_taken_first_count'] / data['retraced_days']) * 100
            low_first_pct = (data['low_taken_first_count'] / data['retraced_days']) * 100
            print(f"\nBEFORE HITTING 50%:")
            print(f"  - Range high taken first: {data['high_taken_first_count']} times ({high_first_pct:.1f}%)")
            print(f"  - Range low taken first: {data['low_taken_first_count']} times ({low_first_pct:.1f}%)")
            
            print(f"\nRETRACEMENT TIMING DURING 9:30AM-4PM (top times):")
            sorted_slots = sorted(data['time_slots'].items(), key=lambda x: x[1], reverse=True)[:10]
            
            print(f"\n{'Time Slot'.ljust(15)} | {'Count'.ljust(10)} | {'% of Retracements'.ljust(20)} | Bar Chart")
            print("-" * 100)
            
            for time_slot, count in sorted_slots:
                pct_of_retracements = (count / data['retraced_days']) * 100
                bar_length = int(pct_of_retracements / 2)
                bar = '█' * bar_length
                print(f"{time_slot.ljust(15)} | {str(count).ljust(10)} | {pct_of_retracements:5.2f}%             | {bar}")

# Display overall results
display_retracement_stats(retracement_stats['overall'], 
                         "OVERALL 50% RANGE RETRACEMENT STATISTICS (9:30AM-4PM) - ALL RANGES SORTED BY SUCCESS RATE",
                         show_all=True)

# Display pattern-specific results
pattern_titles = {
    'london_engulfs_asia': 'LONDON ENGULFS ASIA - 50% Range Retracement Statistics',
    'asia_engulfs_london': 'ASIA ENGULFS LONDON - 50% Range Retracement Statistics',
    'london_partially_engulfs_upwards': 'LONDON PARTIALLY ENGULFS UPWARDS - 50% Range Retracement Statistics',
    'london_partially_engulfs_downwards': 'LONDON PARTIALLY ENGULFS DOWNWARDS - 50% Range Retracement Statistics'
}

for pattern_key, pattern_title in pattern_titles.items():
    # Check if pattern has data
    has_data = any(retracement_stats[pattern_key][range_name]['total_days'] > 0 
                   for range_name in RANGES_TO_TEST.keys())
    if has_data:
        display_retracement_stats(retracement_stats[pattern_key], pattern_title, show_all=True)

print("\n" + "=" * 160)
print("ANALYSIS COMPLETE")
print("=" * 160)

# Export to CSV for further analysis
export_data = []

for pattern in ['overall', 'london_engulfs_asia', 'asia_engulfs_london', 
                'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
    for range_name in RANGES_TO_TEST.keys():
        data = retracement_stats[pattern][range_name]
        if data['total_days'] > 0:
            retracement_pct = (data['retraced_days'] / data['total_days']) * 100
            avg_range = sum(data['avg_range_size']) / len(data['avg_range_size']) if data['avg_range_size'] else 0
            avg_range_pct = sum(data['avg_range_size_pct']) / len(data['avg_range_size_pct']) if data['avg_range_size_pct'] else 0
            high_first_pct = (data['high_taken_first_count'] / data['retraced_days'] * 100) if data['retraced_days'] > 0 else 0
            low_first_pct = (data['low_taken_first_count'] / data['retraced_days'] * 100) if data['retraced_days'] > 0 else 0
            
            # Calculate open position retracement rates
            above_retrace_pct = (data['retraced_when_above'] / data['opened_above_range'] * 100) if data['opened_above_range'] > 0 else 0
            below_retrace_pct = (data['retraced_when_below'] / data['opened_below_range'] * 100) if data['opened_below_range'] > 0 else 0
            inside_retrace_pct = (data['retraced_when_inside'] / data['opened_inside_range'] * 100) if data['opened_inside_range'] > 0 else 0
            
            export_data.append({
                'Pattern': pattern,
                'Range': range_name,
                'Total_Days': data['total_days'],
                'Retraced_Days': data['retraced_days'],
                'Retracement_Pct': retracement_pct,
                'Avg_Range_Size_Pts': avg_range,
                'Avg_Range_Size_Pct': avg_range_pct,
                'High_Taken_First_Pct': high_first_pct,
                'Low_Taken_First_Pct': low_first_pct,
                'Opened_Above_Count': data['opened_above_range'],
                'Opened_Below_Count': data['opened_below_range'],
                'Opened_Inside_Count': data['opened_inside_range'],
                'Retrace_When_Above_Pct': above_retrace_pct,
                'Retrace_When_Below_Pct': below_retrace_pct,
                'Retrace_When_Inside_Pct': inside_retrace_pct,
            })

export_df = pd.DataFrame(export_data)
# Sort by retracement percentage
export_df = export_df.sort_values('Retracement_Pct', ascending=False)
output_path = r"C:\Users\tammm\NQ and ES analysis\enhanced_range_retracement_summary.csv"
export_df.to_csv(output_path, index=False)

print(f"\nSummary exported to: {output_path}")

# Export detailed distance bucket analysis
distance_export_data = []

for pattern in ['overall', 'london_engulfs_asia', 'asia_engulfs_london', 
                'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
    for range_name in RANGES_TO_TEST.keys():
        data = retracement_stats[pattern][range_name]
        if data['total_days'] > 0:
            for i in range(len(DISTANCE_BUCKETS) - 1):
                bucket_data = data['distance_buckets'][i]
                if bucket_data['total'] > 0:
                    bucket_retrace_pct = (bucket_data['retraced'] / bucket_data['total']) * 100
                    
                    if DISTANCE_BUCKETS[i+1] == float('inf'):
                        bucket_label = f"{DISTANCE_BUCKETS[i]:.2f}%+"
                    else:
                        bucket_label = f"{DISTANCE_BUCKETS[i]:.2f}%-{DISTANCE_BUCKETS[i+1]:.2f}%"
                    
                    distance_export_data.append({
                        'Pattern': pattern,
                        'Range': range_name,
                        'Distance_Bucket': bucket_label,
                        'Total_Days': bucket_data['total'],
                        'Retraced_Days': bucket_data['retraced'],
                        'Retracement_Pct': bucket_retrace_pct
                    })

distance_export_df = pd.DataFrame(distance_export_data)
distance_output_path = r"C:\Users\tammm\NQ and ES analysis\distance_bucket_analysis.csv"
distance_export_df.to_csv(distance_output_path, index=False)

print(f"Distance bucket analysis exported to: {distance_output_path}")

# Export day of week analysis
dow_export_data = []
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

for pattern in ['overall', 'london_engulfs_asia', 'asia_engulfs_london', 
                'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
    for range_name in RANGES_TO_TEST.keys():
        data = retracement_stats[pattern][range_name]
        if data['total_days'] > 0:
            for i, day_name in enumerate(days):
                day_data = data['day_of_week'][i]
                if day_data['total'] > 0:
                    day_retrace_pct = (day_data['retraced'] / day_data['total']) * 100
                    
                    dow_export_data.append({
                        'Pattern': pattern,
                        'Range': range_name,
                        'Day_Of_Week': day_name,
                        'Total_Days': day_data['total'],
                        'Retraced_Days': day_data['retraced'],
                        'Retracement_Pct': day_retrace_pct
                    })

dow_export_df = pd.DataFrame(dow_export_data)
dow_output_path = r"C:\Users\tammm\NQ and ES analysis\day_of_week_analysis.csv"
dow_export_df.to_csv(dow_output_path, index=False)

print(f"Day of week analysis exported to: {dow_output_path}")

# Print quick summary
print("\n" + "=" * 160)
print("QUICK SUMMARY - TOP 10 RANGES OVERALL")
print("=" * 160)
overall_export = export_df[export_df['Pattern'] == 'overall'].head(10)
for idx, row in overall_export.iterrows():
    print(f"{row['Range']}: {row['Retracement_Pct']:.2f}% success rate "
          f"({row['Total_Days']} days, avg range: {row['Avg_Range_Size_Pct']:.3f}%, "
          f"above: {row['Retrace_When_Above_Pct']:.1f}%, below: {row['Retrace_When_Below_Pct']:.1f}%, inside: {row['Retrace_When_Inside_Pct']:.1f}%)")

print("\n" + "=" * 160)
print("KEY INSIGHTS BY DISTANCE FROM RANGE")
print("=" * 160)
print("\nShowing top 5 ranges and their retracement rates by distance bucket:")
top_5_ranges = overall_export.head(5)['Range'].tolist()

for range_name in top_5_ranges:
    print(f"\n{range_name}:")
    range_distance_data = distance_export_df[
        (distance_export_df['Pattern'] == 'overall') & 
        (distance_export_df['Range'] == range_name)
    ].sort_values('Distance_Bucket')
    
    for _, row in range_distance_data.iterrows():
        print(f"  {row['Distance_Bucket'].ljust(20)}: {row['Retracement_Pct']:5.1f}% ({row['Retraced_Days']}/{row['Total_Days']} days)")

Processing range retracement data with enhanced tracking...
Processed 500 days...
Processed 1000 days...
Processed 1500 days...
Processed 2000 days...
Processed 2500 days...
Processed 3000 days...
Processed 3500 days...

OVERALL 50% RANGE RETRACEMENT STATISTICS (9:30AM-4PM) - ALL RANGES SORTED BY SUCCESS RATE

Rank   Range                          Sample     Success %    Avg Range %     Open Above   Open Below   Open Inside 
----------------------------------------------------------------------------------------------------------------------------------------------------------------
1      PreMarket_9am-9:30am           3126        97.70% ★★★    0.262%          0.0%         0.0%       100.0%
2      PreMarket_8am-9:30am           3126        91.49% ★★★    0.437%          0.0%         0.0%       100.0%
3      PreMarket_8:30am-9am           3120        87.98% ★★★    0.278%         20.3%        19.7%        60.0%
4      PreMarket_8am-9am              3122        87.03% ★★★    0.350%       

In [2]:
#!/usr/bin/env python
# coding: utf-8

"""
ENHANCED RANGE RETRACEMENT ANALYSIS VISUALIZATIONS
Includes: Open position tracking, distance buckets, day of week analysis, percentage ranges
Add this at the end of your enhanced range retracement notebook
Saves all images to a "range_retr_visualisations" subfolder
"""

import matplotlib.pyplot as plt
import matplotlib
import os
import numpy as np
matplotlib.rcParams['font.family'] = 'monospace'

print("\n" + "=" * 120)
print("CREATING ENHANCED RANGE RETRACEMENT VISUALIZATIONS")
print("=" * 120)

# Create output folder
viz_folder = os.path.join(r"C:\Users\tammm\NQ and ES analysis", 'range_retr_visualisations')
os.makedirs(viz_folder, exist_ok=True)
print(f"\nSaving visualizations to: {viz_folder}")

# ============================================================
# 1. OVERALL RANKINGS TABLE (WITH OPEN POSITION DATA)
# ============================================================

def save_rankings_table(stats, pattern_name, show_all=False):
    """Create ranking table for range retracement success rates with open position data"""
    
    # Prepare data
    range_list = []
    for range_name, data in stats.items():
        if data['total_days'] == 0:
            continue
        
        retracement_pct = (data['retraced_days'] / data['total_days']) * 100
        avg_range_pct = sum(data['avg_range_size_pct']) / len(data['avg_range_size_pct']) if data['avg_range_size_pct'] else 0
        
        # Calculate open position stats
        above_pct = (data['opened_above_range'] / data['total_days'] * 100) if data['total_days'] > 0 else 0
        below_pct = (data['opened_below_range'] / data['total_days'] * 100) if data['total_days'] > 0 else 0
        inside_pct = (data['opened_inside_range'] / data['total_days'] * 100) if data['total_days'] > 0 else 0
        
        range_list.append({
            'name': range_name,
            'total': data['total_days'],
            'retraced': data['retraced_days'],
            'pct': retracement_pct,
            'avg_range_pct': avg_range_pct,
            'above_pct': above_pct,
            'below_pct': below_pct,
            'inside_pct': inside_pct
        })
    
    # Sort by retracement percentage
    range_list.sort(key=lambda x: x['pct'], reverse=True)
    
    if not show_all:
        range_list = range_list[:20]
    
    if len(range_list) == 0:
        print(f"⚠ No data for {pattern_name}")
        return
    
    # Create visualization
    rows_per_page = 25
    num_pages = (len(range_list) + rows_per_page - 1) // rows_per_page
    
    for page in range(num_pages):
        start_idx = page * rows_per_page
        end_idx = min(start_idx + rows_per_page, len(range_list))
        page_data = range_list[start_idx:end_idx]
        
        fig, ax = plt.subplots(figsize=(18, max(8, len(page_data) * 0.45)))
        ax.axis('off')
        
        table_data = []
        for rank, item in enumerate(page_data, start_idx + 1):
            # Stars for visual indicator
            if item['pct'] >= 70:
                stars = "★★★"
            elif item['pct'] >= 60:
                stars = "★★"
            elif item['pct'] >= 50:
                stars = "★"
            else:
                stars = ""
            
            table_data.append([
                f"{rank}",
                item['name'],
                f"{item['total']:,}",
                f"{item['pct']:.2f}%",
                stars,
                f"{item['avg_range_pct']:.3f}%",
                f"{item['above_pct']:.1f}%",
                f"{item['below_pct']:.1f}%",
                f"{item['inside_pct']:.1f}%"
            ])
        
        headers = ['Rank', 'Range', 'Sample', 'Success %', '★', 'Avg Range %', 
                   'Open Above', 'Open Below', 'Open Inside']
        
        table = ax.table(cellText=table_data, colLabels=headers, cellLoc='left', loc='center',
                         colWidths=[0.05, 0.25, 0.08, 0.10, 0.05, 0.12, 0.10, 0.10, 0.10])
        
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1, 1.8)
        
        # Style header
        for i in range(len(headers)):
            table[(0, i)].set_facecolor('#000000')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        # Style rows
        for i in range(1, len(table_data) + 1):
            for j in range(len(headers)):
                if i % 2 == 0:
                    table[(i, j)].set_facecolor('#EEEEEE')
        
        # Title
        title_text = f'RANGES BY 50% RETRACEMENT SUCCESS RATE'
        if num_pages > 1:
            title_text += f' (Page {page+1}/{num_pages})'
        if pattern_name != 'overall':
            title_text = f'{pattern_name.upper().replace("_", " ")} - {title_text}'
        
        plt.title(title_text, fontsize=13, fontweight='bold', family='monospace', pad=15)
        
        # Filename
        page_suffix = f'_page{page+1}' if num_pages > 1 else ''
        filename = os.path.join(viz_folder, f'rankings_{pattern_name}{page_suffix}.png')
        plt.tight_layout()
        plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"✓ rankings_{pattern_name}{page_suffix}.png")

print("\n1. Overall Rankings Table (All Ranges)...")
save_rankings_table(retracement_stats['overall'], 'overall', show_all=True)

# ============================================================
# 2. PATTERN-SPECIFIC RANKINGS
# ============================================================

print("\n2. Pattern-Specific Rankings...")
pattern_names = ['london_engulfs_asia', 'asia_engulfs_london', 
                 'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']

for pattern in pattern_names:
    has_data = any(retracement_stats[pattern][range_name]['total_days'] > 0 
                   for range_name in RANGES_TO_TEST.keys())
    if has_data:
        save_rankings_table(retracement_stats[pattern], pattern, show_all=False)

# ============================================================
# 3. OPEN POSITION ANALYSIS (TOP RANGES)
# ============================================================

def save_open_position_analysis(stats, pattern_name, top_n=15):
    """Create detailed open position analysis for top ranges"""
    
    # Get top ranges
    range_list = []
    for range_name, data in stats.items():
        if data['total_days'] == 0:
            continue
        retracement_pct = (data['retraced_days'] / data['total_days']) * 100
        range_list.append({'name': range_name, 'pct': retracement_pct, 'data': data})
    
    range_list.sort(key=lambda x: x['pct'], reverse=True)
    range_list = range_list[:top_n]
    
    fig, ax = plt.subplots(figsize=(18, max(10, len(range_list) * 0.5)))
    ax.axis('off')
    
    table_data = []
    for item in range_list:
        data = item['data']
        
        # Calculate retracement rates by position
        above_retrace = (data['retraced_when_above'] / data['opened_above_range'] * 100) if data['opened_above_range'] > 0 else 0
        below_retrace = (data['retraced_when_below'] / data['opened_below_range'] * 100) if data['opened_below_range'] > 0 else 0
        inside_retrace = (data['retraced_when_inside'] / data['opened_inside_range'] * 100) if data['opened_inside_range'] > 0 else 0
        
        table_data.append([
            item['name'],
            f"{item['pct']:.1f}%",
            f"{data['opened_above_range']:,}",
            f"{above_retrace:.1f}%",
            f"{data['opened_below_range']:,}",
            f"{below_retrace:.1f}%",
            f"{data['opened_inside_range']:,}",
            f"{inside_retrace:.1f}%"
        ])
    
    headers = ['Range', 'Overall\nSuccess', 'Open Above\nCount', 'Above\nRetrace %',
               'Open Below\nCount', 'Below\nRetrace %', 'Open Inside\nCount', 'Inside\nRetrace %']
    
    table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center',
                     colWidths=[0.25, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10])
    
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 2.0)
    
    # Style header
    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#000000')
        table[(0, i)].set_text_props(weight='bold', color='white', size=7)
    
    # Style rows and highlight best retracement rate
    for i in range(1, len(table_data) + 1):
        # Get retracement percentages for this row
        above_val = float(table_data[i-1][3].rstrip('%'))
        below_val = float(table_data[i-1][5].rstrip('%'))
        inside_val = float(table_data[i-1][7].rstrip('%'))
        max_val = max(above_val, below_val, inside_val)
        
        for j in range(len(headers)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#EEEEEE')
            
            # Highlight best position
            if j == 3 and above_val == max_val and above_val > 0:
                table[(i, j)].set_facecolor('#90EE90')
            elif j == 5 and below_val == max_val and below_val > 0:
                table[(i, j)].set_facecolor('#90EE90')
            elif j == 7 and inside_val == max_val and inside_val > 0:
                table[(i, j)].set_facecolor('#90EE90')
    
    title_text = f'9:30AM OPEN POSITION ANALYSIS - Top {len(range_list)} Ranges'
    if pattern_name != 'overall':
        title_text = f'{pattern_name.upper().replace("_", " ")} - {title_text}'
    
    plt.title(title_text + '\n(Green = Best Retracement Rate for Each Range)', 
              fontsize=12, fontweight='bold', family='monospace', pad=15)
    
    filename = os.path.join(viz_folder, f'open_position_{pattern_name}.png')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ open_position_{pattern_name}.png")

print("\n3. Open Position Analysis...")
save_open_position_analysis(retracement_stats['overall'], 'overall', top_n=15)

# ============================================================
# 4. DISTANCE BUCKET ANALYSIS (TOP RANGES)
# ============================================================

def save_distance_bucket_analysis(stats, pattern_name, top_n=10):
    """Create distance bucket analysis for top ranges"""
    
    # Get top ranges
    range_list = []
    for range_name, data in stats.items():
        if data['total_days'] == 0:
            continue
        retracement_pct = (data['retraced_days'] / data['total_days']) * 100
        range_list.append({'name': range_name, 'pct': retracement_pct, 'data': data})
    
    range_list.sort(key=lambda x: x['pct'], reverse=True)
    
    # Create one visualization per top range
    for idx, item in enumerate(range_list[:top_n]):
        range_name = item['name']
        data = item['data']
        
        fig, ax = plt.subplots(figsize=(14, 10))
        ax.axis('off')
        
        table_data = []
        
        # Add bucket data
        for i in range(len(DISTANCE_BUCKETS) - 1):
            bucket_data = data['distance_buckets'][i]
            
            if bucket_data['total'] > 0:
                bucket_retrace_pct = (bucket_data['retraced'] / bucket_data['total']) * 100
                bar_length = int(bucket_retrace_pct / 2)
                bar = '█' * bar_length
                
                if DISTANCE_BUCKETS[i+1] == float('inf'):
                    bucket_label = f"{DISTANCE_BUCKETS[i]:.2f}%+"
                else:
                    bucket_label = f"{DISTANCE_BUCKETS[i]:.2f}%-{DISTANCE_BUCKETS[i+1]:.2f}%"
                
                table_data.append([
                    bucket_label,
                    f"{bucket_data['total']:,}",
                    f"{bucket_data['retraced']:,}",
                    f"{bucket_retrace_pct:.1f}%",
                    bar
                ])
        
        if not table_data:
            continue
        
        headers = ['Distance from\nMidpoint', 'Total\nDays', 'Retraced\nDays', 'Retracement\n%', 'Visual']
        
        table = ax.table(cellText=table_data, colLabels=headers, cellLoc='left', loc='center',
                         colWidths=[0.20, 0.15, 0.15, 0.15, 0.35])
        
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 2.2)
        
        # Style header
        for i in range(len(headers)):
            table[(0, i)].set_facecolor('#000000')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        # Style rows with color coding based on success rate
        for i in range(1, len(table_data) + 1):
            retrace_pct = float(table_data[i-1][3].rstrip('%'))
            
            for j in range(len(headers)):
                if retrace_pct >= 70:
                    table[(i, j)].set_facecolor('#90EE90')  # Light green
                elif retrace_pct >= 60:
                    table[(i, j)].set_facecolor('#FFFF99')  # Light yellow
                elif i % 2 == 0:
                    table[(i, j)].set_facecolor('#EEEEEE')
        
        title_text = f'DISTANCE FROM RANGE MIDPOINT ANALYSIS\n{range_name} (Overall Success: {item["pct"]:.1f}%)'
        if pattern_name != 'overall':
            title_text = f'{pattern_name.upper().replace("_", " ")}\n' + title_text
        
        plt.title(title_text, fontsize=11, fontweight='bold', family='monospace', pad=15)
        
        safe_range_name = range_name.replace(':', '').replace('-', '_')
        filename = os.path.join(viz_folder, f'distance_{pattern_name}_{safe_range_name}.png')
        plt.tight_layout()
        plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"✓ distance_{pattern_name}_{safe_range_name}.png")

print("\n4. Distance Bucket Analysis (Top 10 Ranges)...")
save_distance_bucket_analysis(retracement_stats['overall'], 'overall', top_n=10)

# ============================================================
# 5. DAY OF WEEK ANALYSIS (TOP RANGES)
# ============================================================

def save_day_of_week_analysis(stats, pattern_name, top_n=15):
    """Create day of week analysis for top ranges"""
    
    # Get top ranges
    range_list = []
    for range_name, data in stats.items():
        if data['total_days'] == 0:
            continue
        retracement_pct = (data['retraced_days'] / data['total_days']) * 100
        range_list.append({'name': range_name, 'pct': retracement_pct, 'data': data})
    
    range_list.sort(key=lambda x: x['pct'], reverse=True)
    range_list = range_list[:top_n]
    
    fig, ax = plt.subplots(figsize=(18, max(10, len(range_list) * 0.5)))
    ax.axis('off')
    
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    table_data = []
    
    for item in range_list:
        data = item['data']
        row = [item['name'], f"{item['pct']:.1f}%"]
        
        for i in range(5):
            day_data = data['day_of_week'][i]
            if day_data['total'] > 0:
                day_retrace_pct = (day_data['retraced'] / day_data['total']) * 100
                row.append(f"{day_retrace_pct:.1f}%\n({day_data['total']})")
            else:
                row.append("N/A")
        
        table_data.append(row)
    
    headers = ['Range', 'Overall'] + days
    
    table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center',
                     colWidths=[0.25, 0.10, 0.13, 0.13, 0.13, 0.13, 0.13])
    
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 2.0)
    
    # Style header
    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#000000')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Style rows and highlight best day
    for i in range(1, len(table_data) + 1):
        # Find best day for this range
        day_values = []
        for j in range(2, 7):  # Days columns
            val_str = table_data[i-1][j]
            if val_str != "N/A":
                val = float(val_str.split('%')[0])
                day_values.append((j, val))
        
        if day_values:
            best_col = max(day_values, key=lambda x: x[1])[0]
        else:
            best_col = None
        
        for j in range(len(headers)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#EEEEEE')
            
            # Highlight best day
            if j == best_col:
                table[(i, j)].set_facecolor('#90EE90')
    
    title_text = f'DAY OF WEEK ANALYSIS - Top {len(range_list)} Ranges\n(Green = Best Day for Each Range)'
    if pattern_name != 'overall':
        title_text = f'{pattern_name.upper().replace("_", " ")} - {title_text}'
    
    plt.title(title_text, fontsize=12, fontweight='bold', family='monospace', pad=15)
    
    filename = os.path.join(viz_folder, f'day_of_week_{pattern_name}.png')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ day_of_week_{pattern_name}.png")

print("\n5. Day of Week Analysis...")
save_day_of_week_analysis(retracement_stats['overall'], 'overall', top_n=15)

# ============================================================
# 6. COMPREHENSIVE BREAKDOWN (TOP 5 RANGES)
# ============================================================

def save_comprehensive_breakdown(stats, pattern_name, range_item):
    """Create comprehensive breakdown for a specific range with all new metrics"""
    
    range_name = range_item['name']
    data = stats[range_name]
    
    if data['total_days'] == 0 or data['retraced_days'] == 0:
        return
    
    fig, ax = plt.subplots(figsize=(16, 14))
    ax.axis('off')
    
    table_data = []
    
    # Summary stats
    retracement_pct = (data['retraced_days'] / data['total_days']) * 100
    avg_range_pct = sum(data['avg_range_size_pct']) / len(data['avg_range_size_pct']) if data['avg_range_size_pct'] else 0
    
    table_data.append(['═══ SUMMARY ═══', '', ''])
    table_data.append(['Total Days Analyzed', f"{data['total_days']:,}", ''])
    table_data.append(['Days with 50% Retracement', f"{data['retraced_days']:,}", f"({retracement_pct:.2f}%)"])
    table_data.append(['Average Range Size', f"{avg_range_pct:.3f}%", 'of price'])
    table_data.append(['', '', ''])
    
    # Open position analysis
    table_data.append(['═══ 9:30AM OPEN POSITION ═══', '', ''])
    
    if data['opened_above_range'] > 0:
        above_retrace = (data['retraced_when_above'] / data['opened_above_range']) * 100
        table_data.append(['Opened Above Range', f"{data['opened_above_range']:,} times", 
                          f"{above_retrace:.1f}% retraced"])
    
    if data['opened_below_range'] > 0:
        below_retrace = (data['retraced_when_below'] / data['opened_below_range']) * 100
        table_data.append(['Opened Below Range', f"{data['opened_below_range']:,} times", 
                          f"{below_retrace:.1f}% retraced"])
    
    if data['opened_inside_range'] > 0:
        inside_retrace = (data['retraced_when_inside'] / data['opened_inside_range']) * 100
        table_data.append(['Opened Inside Range', f"{data['opened_inside_range']:,} times", 
                          f"{inside_retrace:.1f}% retraced"])
    
    table_data.append(['', '', ''])
    
    # Pre-retracement behavior
    high_first_pct = (data['high_taken_first_count'] / data['retraced_days']) * 100
    low_first_pct = (data['low_taken_first_count'] / data['retraced_days']) * 100
    
    table_data.append(['═══ BEFORE HITTING 50% ═══', '', ''])
    table_data.append(['Range High Taken First', f"{data['high_taken_first_count']:,}", f"({high_first_pct:.1f}%)"])
    table_data.append(['Range Low Taken First', f"{data['low_taken_first_count']:,}", f"({low_first_pct:.1f}%)"])
    table_data.append(['', '', ''])
    
    # Top timing slots
    table_data.append(['═══ RETRACEMENT TIMING (9:30AM-4PM) ═══', '', ''])
    sorted_slots = sorted(data['time_slots'].items(), key=lambda x: x[1], reverse=True)[:8]
    
    for time_slot, count in sorted_slots:
        pct_of_retracements = (count / data['retraced_days']) * 100
        bar_length = int(pct_of_retracements / 3)
        bar = '█' * bar_length
        table_data.append([time_slot, f"{count:,}", f"{pct_of_retracements:.2f}% {bar}"])
    
    headers = ['Metric', 'Value', 'Detail']
    
    table = ax.table(cellText=table_data, colLabels=headers, cellLoc='left', loc='center',
                     colWidths=[0.35, 0.25, 0.40])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)
    
    # Style header
    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#000000')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Style rows
    for i in range(1, len(table_data) + 1):
        row = table_data[i-1]
        
        # Section headers
        if '═══' in row[0]:
            for j in range(len(headers)):
                table[(i, j)].set_facecolor('#333333')
                table[(i, j)].set_text_props(weight='bold', color='white', size=9)
        # Spacer rows
        elif row[0] == '':
            for j in range(len(headers)):
                table[(i, j)].set_facecolor('#FFFFFF')
        # Data rows
        else:
            for j in range(len(headers)):
                if i % 2 == 0:
                    table[(i, j)].set_facecolor('#EEEEEE')
    
    title_text = f'COMPREHENSIVE BREAKDOWN: {range_name}'
    if pattern_name != 'overall':
        title_text = f'{pattern_name.upper().replace("_", " ")} - {title_text}'
    
    plt.title(title_text, fontsize=12, fontweight='bold', family='monospace', pad=15)
    
    safe_range_name = range_name.replace(':', '').replace('-', '_')
    filename = os.path.join(viz_folder, f'comprehensive_{pattern_name}_{safe_range_name}.png')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    return os.path.basename(filename)

print("\n6. Comprehensive Breakdowns (Top 5 Ranges)...")

# Get top 5 overall ranges
overall_range_list = []
for range_name, data in retracement_stats['overall'].items():
    if data['total_days'] == 0:
        continue
    retracement_pct = (data['retraced_days'] / data['total_days']) * 100
    overall_range_list.append({'name': range_name, 'pct': retracement_pct})

overall_range_list.sort(key=lambda x: x['pct'], reverse=True)

for item in overall_range_list[:5]:
    result = save_comprehensive_breakdown(retracement_stats['overall'], 'overall', item)
    if result:
        print(f"✓ {result}")

# ============================================================
# 7. PATTERN COMPARISON SUMMARY
# ============================================================

def save_pattern_comparison():
    """Create comparison table across all patterns"""
    
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.axis('off')
    
    table_data = []
    
    # Get top 10 overall ranges
    overall_range_list = []
    for range_name, data in retracement_stats['overall'].items():
        if data['total_days'] == 0:
            continue
        retracement_pct = (data['retraced_days'] / data['total_days']) * 100
        overall_range_list.append({'name': range_name, 'pct': retracement_pct})
    
    overall_range_list.sort(key=lambda x: x['pct'], reverse=True)
    
    for item in overall_range_list[:10]:
        range_name = item['name']
        row = [range_name]
        
        # Overall
        overall_data = retracement_stats['overall'][range_name]
        if overall_data['total_days'] > 0:
            overall_pct = (overall_data['retraced_days'] / overall_data['total_days']) * 100
            row.append(f"{overall_pct:.1f}%\n({overall_data['total_days']})")
        else:
            row.append("N/A")
        
        # Each pattern
        for pattern in ['london_engulfs_asia', 'asia_engulfs_london', 
                       'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
            pattern_data = retracement_stats[pattern][range_name]
            if pattern_data['total_days'] > 0:
                pattern_pct = (pattern_data['retraced_days'] / pattern_data['total_days']) * 100
                row.append(f"{pattern_pct:.1f}%\n({pattern_data['total_days']})")
            else:
                row.append("N/A")
        
        table_data.append(row)
    
    headers = ['Range', 'Overall', 'London\nEngulfs Asia', 'Asia\nEngulfs London', 
               'London Up\nPartial', 'London Down\nPartial']
    
    table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center',
                     colWidths=[0.25, 0.15, 0.15, 0.15, 0.15, 0.15])
    
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 2.5)
    
    # Style header
    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#000000')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Style rows
    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#EEEEEE')
    
    plt.title('PATTERN COMPARISON - Top 10 Ranges\n(Success % and Sample Size)',
              fontsize=13, fontweight='bold', family='monospace', pad=15)
    
    filename = os.path.join(viz_folder, 'pattern_comparison.png')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ pattern_comparison.png")

print("\n7. Pattern Comparison Summary...")
save_pattern_comparison()

# ============================================================
# 8. DISTANCE BUCKET HEATMAP (OVERALL)
# ============================================================

def save_distance_heatmap_overall():
    """Create heatmap showing distance bucket performance across top ranges"""
    
    # Get top 10 ranges
    range_list = []
    for range_name, data in retracement_stats['overall'].items():
        if data['total_days'] == 0:
            continue
        retracement_pct = (data['retraced_days'] / data['total_days']) * 100
        range_list.append({'name': range_name, 'pct': retracement_pct, 'data': data})
    
    range_list.sort(key=lambda x: x['pct'], reverse=True)
    range_list = range_list[:10]
    
    fig, ax = plt.subplots(figsize=(18, 12))
    ax.axis('off')
    
    # Build table
    table_data = []
    bucket_headers = []
    
    for i in range(len(DISTANCE_BUCKETS) - 1):
        if DISTANCE_BUCKETS[i+1] == float('inf'):
            bucket_headers.append(f"{DISTANCE_BUCKETS[i]:.2f}%+")
        else:
            bucket_headers.append(f"{DISTANCE_BUCKETS[i]:.2f}%-\n{DISTANCE_BUCKETS[i+1]:.2f}%")
    
    for item in range_list:
        data = item['data']
        row = [item['name'], f"{item['pct']:.1f}%"]
        
        for i in range(len(DISTANCE_BUCKETS) - 1):
            bucket_data = data['distance_buckets'][i]
            if bucket_data['total'] > 0:
                bucket_pct = (bucket_data['retraced'] / bucket_data['total']) * 100
                row.append(f"{bucket_pct:.1f}%\n({bucket_data['total']})")
            else:
                row.append("—")
        
        table_data.append(row)
    
    headers = ['Range', 'Overall'] + bucket_headers
    
    table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center',
                     colWidths=[0.20, 0.08] + [0.072] * len(bucket_headers))
    
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 2.2)
    
    # Style header
    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#000000')
        table[(0, i)].set_text_props(weight='bold', color='white', size=7)
    
    # Style with color gradient
    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            if j <= 1:  # Range name and overall
                if i % 2 == 0:
                    table[(i, j)].set_facecolor('#EEEEEE')
            else:  # Distance buckets
                val_str = table_data[i-1][j]
                if val_str != "—":
                    val = float(val_str.split('%')[0])
                    
                    # Color coding
                    if val >= 70:
                        table[(i, j)].set_facecolor('#006400')  # Dark green
                        table[(i, j)].set_text_props(color='white')
                    elif val >= 60:
                        table[(i, j)].set_facecolor('#90EE90')  # Light green
                    elif val >= 50:
                        table[(i, j)].set_facecolor('#FFFF99')  # Yellow
                    elif val >= 40:
                        table[(i, j)].set_facecolor('#FFD700')  # Gold
                    else:
                        if i % 2 == 0:
                            table[(i, j)].set_facecolor('#EEEEEE')
    
    plt.title('DISTANCE BUCKET HEATMAP - Top 10 Ranges\n(Dark Green ≥70%, Light Green ≥60%, Yellow ≥50%)',
              fontsize=12, fontweight='bold', family='monospace', pad=15)
    
    filename = os.path.join(viz_folder, 'distance_heatmap.png')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ distance_heatmap.png")

print("\n8. Distance Bucket Heatmap...")
save_distance_heatmap_overall()

# ============================================================
# 9. BEST COMBINATIONS SUMMARY
# ============================================================

def save_best_combinations():
    """Identify and display best combinations of range + open position + distance"""
    
    combinations = []
    
    # Analyze top 15 ranges
    range_list = []
    for range_name, data in retracement_stats['overall'].items():
        if data['total_days'] == 0:
            continue
        retracement_pct = (data['retraced_days'] / data['total_days']) * 100
        range_list.append({'name': range_name, 'pct': retracement_pct, 'data': data})
    
    range_list.sort(key=lambda x: x['pct'], reverse=True)
    
    for item in range_list[:15]:
        range_name = item['name']
        data = item['data']
        
        # Check each open position
        positions = [
            ('Above', data['opened_above_range'], data['retraced_when_above']),
            ('Below', data['opened_below_range'], data['retraced_when_below']),
            ('Inside', data['opened_inside_range'], data['retraced_when_inside'])
        ]
        
        for pos_name, total, retraced in positions:
            if total >= 20:  # Minimum sample size
                pos_pct = (retraced / total) * 100
                
                # Check distance buckets for this position (simplified - using overall buckets)
                for i in range(len(DISTANCE_BUCKETS) - 1):
                    bucket_data = data['distance_buckets'][i]
                    if bucket_data['total'] >= 10:
                        bucket_pct = (bucket_data['retraced'] / bucket_data['total']) * 100
                        
                        if DISTANCE_BUCKETS[i+1] == float('inf'):
                            bucket_label = f"{DISTANCE_BUCKETS[i]:.2f}%+"
                        else:
                            bucket_label = f"{DISTANCE_BUCKETS[i]:.2f}%-{DISTANCE_BUCKETS[i+1]:.2f}%"
                        
                        combinations.append({
                            'range': range_name,
                            'position': pos_name,
                            'distance': bucket_label,
                            'pos_success': pos_pct,
                            'bucket_success': bucket_pct,
                            'avg_success': (pos_pct + bucket_pct) / 2,
                            'pos_sample': total,
                            'bucket_sample': bucket_data['total']
                        })
    
    # Sort by average success
    combinations.sort(key=lambda x: x['avg_success'], reverse=True)
    combinations = combinations[:20]
    
    fig, ax = plt.subplots(figsize=(18, 12))
    ax.axis('off')
    
    table_data = []
    for rank, combo in enumerate(combinations, 1):
        stars = "★★★" if combo['avg_success'] >= 70 else "★★" if combo['avg_success'] >= 65 else "★"
        
        table_data.append([
            f"{rank}",
            combo['range'],
            combo['position'],
            combo['distance'],
            f"{combo['pos_success']:.1f}%",
            f"({combo['pos_sample']})",
            f"{combo['bucket_success']:.1f}%",
            f"({combo['bucket_sample']})",
            f"{combo['avg_success']:.1f}%",
            stars
        ])
    
    headers = ['Rank', 'Range', 'Open\nPosition', 'Distance\nBucket', 
               'Position\nSuccess', 'Pos\nSample', 'Distance\nSuccess', 'Dist\nSample',
               'Combined\nScore', '★']
    
    table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center',
                     colWidths=[0.05, 0.22, 0.10, 0.12, 0.10, 0.08, 0.10, 0.08, 0.10, 0.05])
    
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.8)
    
    # Style header
    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#000000')
        table[(0, i)].set_text_props(weight='bold', color='white', size=7)
    
    # Style rows
    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#EEEEEE')
    
    plt.title('BEST COMBINATIONS - Range + Open Position + Distance\n(Top 20 by Combined Success Score)',
              fontsize=12, fontweight='bold', family='monospace', pad=15)
    
    filename = os.path.join(viz_folder, 'best_combinations.png')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ best_combinations.png")

print("\n9. Best Combinations Analysis...")
save_best_combinations()

print("\n" + "=" * 120)
print(f"✓ DONE! All enhanced visualizations saved to:")
print(f"  {viz_folder}")
print("\nFiles created:")
print("  • Overall rankings (all ranges, possibly multiple pages)")
print("  • Pattern-specific rankings (4 patterns)")
print("  • Open position analysis")
print("  • Distance bucket analysis (10 detailed charts)")
print("  • Day of week analysis")
print("  • Comprehensive breakdowns (top 5 ranges)")
print("  • Pattern comparison summary")
print("  • Distance bucket heatmap")
print("  • Best combinations analysis")
print("\nTotal: ~30+ visualization files")
print("\nEach visualization includes:")
print("  ✓ Percentage-based range sizes")
print("  ✓ Open position tracking (above/below/inside range)")
print("  ✓ Distance from range midpoint analysis")
print("  ✓ Day of week performance")
print("  ✓ Sample sizes for statistical confidence")
print("  ✓ Color-coded success rates")
print("=" * 120)


CREATING ENHANCED RANGE RETRACEMENT VISUALIZATIONS

Saving visualizations to: C:\Users\tammm\NQ and ES analysis\range_retr_visualisations

1. Overall Rankings Table (All Ranges)...
✓ rankings_overall_page1.png
✓ rankings_overall_page2.png

2. Pattern-Specific Rankings...
✓ rankings_london_engulfs_asia.png
✓ rankings_asia_engulfs_london.png
✓ rankings_london_partially_engulfs_upwards.png
✓ rankings_london_partially_engulfs_downwards.png

3. Open Position Analysis...
✓ open_position_overall.png

4. Distance Bucket Analysis (Top 10 Ranges)...
✓ distance_overall_PreMarket_9am_930am.png
✓ distance_overall_PreMarket_8am_930am.png
✓ distance_overall_PreMarket_830am_9am.png
✓ distance_overall_PreMarket_8am_9am.png
✓ distance_overall_PreMarket_8am_830am.png
✓ distance_overall_PreMarket_730am_8am.png
✓ distance_overall_PreMarket_7am_8am.png
✓ distance_overall_PreMarket_7am_730am.png
✓ distance_overall_PreMarket_630am_7am.png
✓ distance_overall_PreMarket_6am_7am.png

5. Day of Week Analysis...
✓

In [3]:
## Distance from Range Midpoint vs 50% Retracement Success Rate
## Answers: "If price opens too far from a range, are retracement odds reduced?"

import pandas as pd
from collections import defaultdict
import numpy as np

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data from 2013 to 2025
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

# Adjust the index to group by NY trading day (18:00 to 17:00)
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

# Define ranges to test
RANGES_TO_TEST = {
    # Previous day ranges
    'Globex_6pm-7pm': (18, 0, 19, 0, True),
    'Globex_6pm-8pm': (18, 0, 20, 0, True),
    'Asia_8pm-9pm': (20, 0, 21, 0, True),
    'Asia_8pm-10pm': (20, 0, 22, 0, True),
    'Asia_10pm-11pm': (22, 0, 23, 0, True),
    'Asia_11pm-12am': (23, 0, 0, 0, True),
    
    # Current day ranges - early morning
    'Midnight_12am-1am': (0, 0, 1, 0, False),
    'Late_Night_1am-2am': (1, 0, 2, 0, False),
    'London_2am-3am': (2, 0, 3, 0, False),
    'London_3am-4am': (3, 0, 4, 0, False),
    'London_4am-5am': (4, 0, 5, 0, False),
    'London_5am-6am': (5, 0, 6, 0, False),
    
    # Pre-market ranges
    'PreMarket_6am-7am': (6, 0, 7, 0, False),
    'PreMarket_7am-8am': (7, 0, 8, 0, False),
    'PreMarket_8am-9am': (8, 0, 9, 0, False),
    'PreMarket_8am-9:30am': (8, 0, 9, 30, False),
    'PreMarket_9am-9:30am': (9, 0, 9, 30, False),
}

# Distance buckets (in percentage from range midpoint)
# These represent how far 9:30am open is from the range's 50% level
DISTANCE_BUCKETS = [
    (0, 0.1, "0.0%-0.1% (Nearly at midpoint)"),
    (0.1, 0.25, "0.1%-0.25%"),
    (0.25, 0.5, "0.25%-0.5%"),
    (0.5, 0.75, "0.5%-0.75%"),
    (0.75, 1.0, "0.75%-1.0%"),
    (1.0, 1.5, "1.0%-1.5%"),
    (1.5, 2.0, "1.5%-2.0%"),
    (2.0, 3.0, "2.0%-3.0%"),
    (3.0, 5.0, "3.0%-5.0%"),
    (5.0, float('inf'), "5.0%+ (Very far)")
]

print("=" * 120)
print("DISTANCE FROM RANGE MIDPOINT vs 50% RETRACEMENT SUCCESS RATE")
print("=" * 120)
print("\nQuestion: If 9:30am opens far from a range's midpoint, does it still retrace to the 50% level?")
print("\nProcessing data...")

# Initialize tracking
range_stats = {}

for range_name in RANGES_TO_TEST.keys():
    range_stats[range_name] = {
        'total_days': 0,
        'distance_buckets': {i: {'total': 0, 'retraced': 0, 'distances': []} for i in range(len(DISTANCE_BUCKETS))},
        'by_direction': {
            'above': {i: {'total': 0, 'retraced': 0} for i in range(len(DISTANCE_BUCKETS))},
            'below': {i: {'total': 0, 'retraced': 0} for i in range(len(DISTANCE_BUCKETS))}
        }
    }

processed_count = 0

for date in dates[1:]:
    processed_count += 1
    if processed_count % 500 == 0:
        print(f"Processed {processed_count} days...")
    
    actual_date = date + pd.Timedelta(hours=7)
    
    # Get 9:30am open price
    open_930_time = actual_date.replace(hour=9, minute=30, second=0)
    open_930_data = df.loc[open_930_time:open_930_time]
    
    if open_930_data.empty:
        continue
    
    open_930_price = open_930_data.iloc[0]['Open']
    
    # Get the 9:30am-4pm session data for retracement checks
    retracement_session_start = actual_date.replace(hour=9, minute=30, second=0)
    retracement_session_end = actual_date.replace(hour=16, minute=0, second=0)
    retracement_session_data = df.loc[retracement_session_start:retracement_session_end]
    
    if retracement_session_data.empty:
        continue
    
    # Process each range
    for range_name, (start_hour, start_min, end_hour, end_min, is_prev_day) in RANGES_TO_TEST.items():
        # Determine the actual start and end times
        if is_prev_day:
            range_start = actual_date.replace(hour=start_hour, minute=start_min, second=0) - pd.Timedelta(days=1)
        else:
            range_start = actual_date.replace(hour=start_hour, minute=start_min, second=0)
        
        if is_prev_day and end_hour == 0:
            range_end = actual_date.replace(hour=end_hour, minute=end_min, second=0)
        elif is_prev_day:
            range_end = actual_date.replace(hour=end_hour, minute=end_min, second=0) - pd.Timedelta(days=1)
        else:
            range_end = actual_date.replace(hour=end_hour, minute=end_min, second=0)
        
        # Get the range data
        range_data = df.loc[range_start:range_end]
        
        if range_data.empty:
            continue
        
        # Calculate range high, low, and midpoint
        range_high = range_data['High'].max()
        range_low = range_data['Low'].min()
        
        if pd.isna(range_high) or pd.isna(range_low):
            continue
        
        range_size = range_high - range_low
        
        if range_size < 1:
            continue
        
        # Calculate range midpoint (this is the 50% retracement level)
        range_midpoint = (range_high + range_low) / 2
        
        # Calculate distance from 9:30am open to range midpoint
        distance_from_midpoint = abs(open_930_price - range_midpoint)
        distance_pct = (distance_from_midpoint / range_midpoint) * 100
        
        # Determine if 9:30am opened above or below the midpoint
        if open_930_price > range_midpoint:
            direction = 'above'
        else:
            direction = 'below'
        
        # Find which distance bucket this falls into
        bucket_idx = None
        for i, (min_dist, max_dist, label) in enumerate(DISTANCE_BUCKETS):
            if min_dist <= distance_pct < max_dist:
                bucket_idx = i
                break
        
        if bucket_idx is None:
            bucket_idx = len(DISTANCE_BUCKETS) - 1  # Last bucket (5%+)
        
        # Check if 50% level (midpoint) was hit during 9:30am-4pm session
        was_retraced = False
        for idx, row in retracement_session_data.iterrows():
            if row['Low'] <= range_midpoint <= row['High']:
                was_retraced = True
                break
        
        # Update statistics
        range_stats[range_name]['total_days'] += 1
        range_stats[range_name]['distance_buckets'][bucket_idx]['total'] += 1
        range_stats[range_name]['distance_buckets'][bucket_idx]['distances'].append(distance_pct)
        range_stats[range_name]['by_direction'][direction][bucket_idx]['total'] += 1
        
        if was_retraced:
            range_stats[range_name]['distance_buckets'][bucket_idx]['retraced'] += 1
            range_stats[range_name]['by_direction'][direction][bucket_idx]['retraced'] += 1

# ========================================================================
# DISPLAY RESULTS
# ========================================================================

print("\n" + "=" * 120)
print("RESULTS - SORTED BY OVERALL RETRACEMENT SUCCESS RATE")
print("=" * 120)

# Calculate overall success rate for each range for sorting
range_success = []
for range_name, stats in range_stats.items():
    if stats['total_days'] > 0:
        total_retraced = sum(bucket['retraced'] for bucket in stats['distance_buckets'].values())
        success_rate = (total_retraced / stats['total_days']) * 100
        range_success.append((range_name, success_rate, stats))

range_success.sort(key=lambda x: x[1], reverse=True)

# Display top 10 ranges in detail
print("\n" + "=" * 120)
print("TOP 10 RANGES - DETAILED DISTANCE ANALYSIS")
print("=" * 120)

for rank, (range_name, overall_success, stats) in enumerate(range_success[:10], 1):
    print(f"\n{'=' * 120}")
    print(f"#{rank}: {range_name}")
    print(f"Overall Success Rate: {overall_success:.2f}% ({stats['total_days']} days)")
    print(f"{'=' * 120}")
    
    print(f"\n{'Distance from Midpoint':<35} {'Sample':<10} {'Retraced':<12} {'Success %':<12} {'Avg Distance':<15}")
    print("-" * 120)
    
    for i, (min_dist, max_dist, label) in enumerate(DISTANCE_BUCKETS):
        bucket = stats['distance_buckets'][i]
        
        if bucket['total'] > 0:
            success_pct = (bucket['retraced'] / bucket['total']) * 100
            avg_distance = np.mean(bucket['distances']) if bucket['distances'] else 0
            
            # Color indicator
            if success_pct >= 70:
                indicator = "★★★"
            elif success_pct >= 60:
                indicator = "★★"
            elif success_pct >= 50:
                indicator = "★"
            else:
                indicator = ""
            
            print(f"{label:<35} {bucket['total']:<10} {bucket['retraced']:<12} {success_pct:>6.2f}% {indicator:<5} {avg_distance:>6.3f}%")
    
    # Show breakdown by direction (above vs below midpoint)
    print(f"\n{'Breakdown by Direction:':<35}")
    print(f"{'  (Above/Below Midpoint)':<35} {'Sample':<10} {'Retraced':<12} {'Success %':<12}")
    print("-" * 120)
    
    for direction in ['above', 'below']:
        total_dir = sum(bucket['total'] for bucket in stats['by_direction'][direction].values())
        retraced_dir = sum(bucket['retraced'] for bucket in stats['by_direction'][direction].values())
        
        if total_dir > 0:
            success_dir = (retraced_dir / total_dir) * 100
            dir_label = "  Opened ABOVE midpoint" if direction == 'above' else "  Opened BELOW midpoint"
            print(f"{dir_label:<35} {total_dir:<10} {retraced_dir:<12} {success_dir:>6.2f}%")

# ========================================================================
# SUMMARY TABLE - ALL RANGES
# ========================================================================

print("\n" + "=" * 120)
print("SUMMARY - ALL RANGES (Key Distance Buckets)")
print("=" * 120)

print(f"\n{'Rank':<6} {'Range':<30} {'Overall':<10} {'0-0.5%':<10} {'0.5-1%':<10} {'1-2%':<10} {'2-5%':<10} {'5%+':<10}")
print("-" * 120)

for rank, (range_name, overall_success, stats) in enumerate(range_success, 1):
    row = [f"{rank}", range_name, f"{overall_success:.1f}%"]
    
    # Key buckets to show
    key_buckets = [
        [0, 1, 2],  # 0-0.5% (combine first 3 buckets)
        [3],        # 0.5-0.75%
        [4, 5, 6],  # 0.75-2%
        [7, 8],     # 2-5%
        [9]         # 5%+
    ]
    
    for bucket_group in key_buckets:
        total = sum(stats['distance_buckets'][i]['total'] for i in bucket_group)
        retraced = sum(stats['distance_buckets'][i]['retraced'] for i in bucket_group)
        
        if total > 0:
            pct = (retraced / total) * 100
            row.append(f"{pct:.1f}%")
        else:
            row.append("N/A")
    
    print(f"{row[0]:<6} {row[1]:<30} {row[2]:<10} {row[3]:<10} {row[4]:<10} {row[5]:<10} {row[6]:<10} {row[7]:<10}")

# ========================================================================
# EXPORT TO CSV
# ========================================================================

export_data = []

for range_name, stats in range_stats.items():
    if stats['total_days'] > 0:
        for i, (min_dist, max_dist, label) in enumerate(DISTANCE_BUCKETS):
            bucket = stats['distance_buckets'][i]
            
            if bucket['total'] > 0:
                success_pct = (bucket['retraced'] / bucket['total']) * 100
                avg_distance = np.mean(bucket['distances']) if bucket['distances'] else 0
                
                # Direction breakdown
                above_total = stats['by_direction']['above'][i]['total']
                above_retraced = stats['by_direction']['above'][i]['retraced']
                above_pct = (above_retraced / above_total * 100) if above_total > 0 else 0
                
                below_total = stats['by_direction']['below'][i]['total']
                below_retraced = stats['by_direction']['below'][i]['retraced']
                below_pct = (below_retraced / below_total * 100) if below_total > 0 else 0
                
                export_data.append({
                    'Range': range_name,
                    'Distance_Bucket': label,
                    'Min_Distance_Pct': min_dist,
                    'Max_Distance_Pct': max_dist if max_dist != float('inf') else 999,
                    'Total_Days': bucket['total'],
                    'Retraced_Days': bucket['retraced'],
                    'Success_Rate_Pct': success_pct,
                    'Avg_Distance_Pct': avg_distance,
                    'Above_Midpoint_Total': above_total,
                    'Above_Midpoint_Retraced': above_retraced,
                    'Above_Midpoint_Success_Pct': above_pct,
                    'Below_Midpoint_Total': below_total,
                    'Below_Midpoint_Retraced': below_retraced,
                    'Below_Midpoint_Success_Pct': below_pct
                })

export_df = pd.DataFrame(export_data)
export_df = export_df.sort_values(['Range', 'Min_Distance_Pct'])
output_path = r"C:\Users\tammm\NQ and ES analysis\distance_vs_retracement_analysis.csv"
export_df.to_csv(output_path, index=False)

print(f"\nDetailed data exported to: {output_path}")

# ========================================================================
# KEY INSIGHTS
# ========================================================================

print("\n" + "=" * 120)
print("KEY INSIGHTS")
print("=" * 120)

# Find ranges where distance matters most
print("\nRanges where distance from midpoint REDUCES retracement odds:")
print("(Comparing close distance <0.5% vs far distance >2%)")
print("-" * 120)

distance_impact = []

for range_name, stats in range_stats.items():
    # Close buckets (0-0.5%)
    close_total = sum(stats['distance_buckets'][i]['total'] for i in [0, 1, 2])
    close_retraced = sum(stats['distance_buckets'][i]['retraced'] for i in [0, 1, 2])
    close_pct = (close_retraced / close_total * 100) if close_total >= 20 else None
    
    # Far buckets (>2%)
    far_total = sum(stats['distance_buckets'][i]['total'] for i in [7, 8, 9])
    far_retraced = sum(stats['distance_buckets'][i]['retraced'] for i in [7, 8, 9])
    far_pct = (far_retraced / far_total * 100) if far_total >= 20 else None
    
    if close_pct is not None and far_pct is not None:
        impact = close_pct - far_pct
        distance_impact.append((range_name, close_pct, far_pct, impact, close_total, far_total))

distance_impact.sort(key=lambda x: x[3], reverse=True)

print(f"\n{'Range':<30} {'Close (<0.5%)':<15} {'Far (>2%)':<15} {'Impact':<12} {'Samples':<15}")
print("-" * 120)

for range_name, close_pct, far_pct, impact, close_total, far_total in distance_impact[:10]:
    print(f"{range_name:<30} {close_pct:>6.1f}%         {far_pct:>6.1f}%         {impact:>+6.1f}%     {close_total}/{far_total}")

print("\n" + "=" * 120)
print("ANALYSIS COMPLETE")
print("=" * 120)
print("\nConclusion:")
print("• Positive 'Impact' = Distance reduces retracement odds (avoid when far from midpoint)")
print("• Negative 'Impact' = Distance doesn't matter or helps (trade regardless of distance)")
print("• Look for ranges with high impact AND good close-distance success rates")

DISTANCE FROM RANGE MIDPOINT vs 50% RETRACEMENT SUCCESS RATE

Question: If 9:30am opens far from a range's midpoint, does it still retrace to the 50% level?

Processing data...
Processed 500 days...
Processed 1000 days...
Processed 1500 days...
Processed 2000 days...
Processed 2500 days...
Processed 3000 days...
Processed 3500 days...

RESULTS - SORTED BY OVERALL RETRACEMENT SUCCESS RATE

TOP 10 RANGES - DETAILED DISTANCE ANALYSIS

#1: PreMarket_9am-9:30am
Overall Success Rate: 97.70% (3130 days)

Distance from Midpoint              Sample     Retraced     Success %    Avg Distance   
------------------------------------------------------------------------------------------------------------------------
0.0%-0.1% (Nearly at midpoint)      2808       2755          98.11% ★★★    0.034%
0.1%-0.25%                          294        278           94.56% ★★★    0.142%
0.25%-0.5%                          25         22            88.00% ★★★    0.322%
0.5%-0.75%                          1    

In [4]:
"""
VISUALIZATIONS FOR DISTANCE FROM MIDPOINT ANALYSIS
Shows how distance from range affects 50% retracement success
"""

import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import os
import numpy as np

matplotlib.rcParams['font.family'] = 'monospace'

print("\n" + "=" * 120)
print("CREATING DISTANCE vs RETRACEMENT VISUALIZATIONS")
print("=" * 120)

# Create output folder
viz_folder = os.path.join(r"C:\Users\tammm\NQ and ES analysis", 'distance_analysis_visuals')
os.makedirs(viz_folder, exist_ok=True)
print(f"\nSaving visualizations to: {viz_folder}")

# Load the exported data
data_path = r"C:\Users\tammm\NQ and ES analysis\distance_vs_retracement_analysis.csv"

if not os.path.exists(data_path):
    print(f"\n❌ ERROR: Data file not found at {data_path}")
    print("Please run distance_midpoint_focused_analysis.py first!")
    exit(1)

df = pd.read_csv(data_path)
print(f"✓ Loaded {len(df)} records")

# ============================================================
# 1. LINE CHARTS - Success Rate vs Distance (Top Ranges)
# ============================================================

def create_distance_curve(range_name, range_data):
    """Create line chart showing how success rate changes with distance"""
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Prepare data
    distances = []
    success_rates = []
    sample_sizes = []
    
    for _, row in range_data.iterrows():
        # Use midpoint of bucket for x-axis
        mid_distance = (row['Min_Distance_Pct'] + min(row['Max_Distance_Pct'], 10)) / 2
        distances.append(mid_distance)
        success_rates.append(row['Success_Rate_Pct'])
        sample_sizes.append(row['Total_Days'])
    
    # Plot line with markers
    ax.plot(distances, success_rates, 'o-', linewidth=2, markersize=8, color='#2E86AB')
    
    # Add sample size labels on points
    for x, y, n in zip(distances, success_rates, sample_sizes):
        ax.annotate(f'n={n}', (x, y), textcoords="offset points", xytext=(0,10), 
                   ha='center', fontsize=8, color='#555')
    
    # Reference lines
    ax.axhline(y=70, color='green', linestyle='--', alpha=0.3, label='70% (Strong)')
    ax.axhline(y=60, color='orange', linestyle='--', alpha=0.3, label='60% (Good)')
    ax.axhline(y=50, color='red', linestyle='--', alpha=0.3, label='50% (Weak)')
    
    ax.set_xlabel('Distance from Range Midpoint (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('50% Retracement Success Rate (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Distance vs Retracement Success: {range_name}', 
                fontsize=14, fontweight='bold', pad=20)
    
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')
    ax.set_ylim(0, 100)
    
    # Format x-axis
    ax.set_xlim(0, max(distances) * 1.1)
    
    safe_name = range_name.replace(':', '').replace('-', '_')
    filename = os.path.join(viz_folder, f'curve_{safe_name}.png')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    return filename

print("\n1. Creating Distance Curves for Top 10 Ranges...")

# Get top 10 ranges by overall success rate
range_success = df.groupby('Range').agg({
    'Total_Days': 'sum',
    'Retraced_Days': 'sum'
}).reset_index()

range_success['Success_Rate'] = (range_success['Retraced_Days'] / range_success['Total_Days']) * 100
range_success = range_success.sort_values('Success_Rate', ascending=False)

for i, range_name in enumerate(range_success['Range'].head(10), 1):
    range_data = df[df['Range'] == range_name].sort_values('Min_Distance_Pct')
    filename = create_distance_curve(range_name, range_data)
    print(f"  ✓ {os.path.basename(filename)}")

# ============================================================
# 2. HEATMAP - All Ranges vs Distance Buckets
# ============================================================

print("\n2. Creating Distance Heatmap...")

# Pivot data for heatmap
heatmap_data = df.pivot_table(
    values='Success_Rate_Pct',
    index='Range',
    columns='Distance_Bucket',
    aggfunc='first'
)

# Sort by overall success rate
overall_success = df.groupby('Range').apply(
    lambda x: (x['Retraced_Days'].sum() / x['Total_Days'].sum()) * 100
).sort_values(ascending=False)

heatmap_data = heatmap_data.loc[overall_success.index]

# Take top 20 ranges
heatmap_data = heatmap_data.head(20)

fig, ax = plt.subplots(figsize=(18, 12))

# Create heatmap
im = ax.imshow(heatmap_data.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

# Set ticks
ax.set_xticks(np.arange(len(heatmap_data.columns)))
ax.set_yticks(np.arange(len(heatmap_data.index)))
ax.set_xticklabels(heatmap_data.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(heatmap_data.index, fontsize=9)

# Add values in cells
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        value = heatmap_data.values[i, j]
        if not np.isnan(value):
            text_color = 'white' if value < 50 else 'black'
            ax.text(j, i, f'{value:.0f}%', ha='center', va='center', 
                   color=text_color, fontsize=7, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Success Rate (%)', rotation=270, labelpad=20, fontsize=11, fontweight='bold')

ax.set_title('Retracement Success Rate by Distance from Midpoint\n(Top 20 Ranges)', 
            fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Distance from Range Midpoint', fontsize=11, fontweight='bold')
ax.set_ylabel('Range', fontsize=11, fontweight='bold')

plt.tight_layout()
filename = os.path.join(viz_folder, 'heatmap_all_ranges.png')
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"  ✓ heatmap_all_ranges.png")

# ============================================================
# 3. BAR CHART - Impact of Distance (Close vs Far)
# ============================================================

print("\n3. Creating Distance Impact Chart...")

# Calculate impact for each range
impact_data = []

for range_name in df['Range'].unique():
    range_df = df[df['Range'] == range_name]
    
    # Close distance (0-0.5%)
    close_df = range_df[range_df['Max_Distance_Pct'] <= 0.5]
    if len(close_df) > 0:
        close_total = close_df['Total_Days'].sum()
        close_retraced = close_df['Retraced_Days'].sum()
        close_rate = (close_retraced / close_total * 100) if close_total >= 20 else None
    else:
        close_rate = None
    
    # Far distance (>2%)
    far_df = range_df[range_df['Min_Distance_Pct'] >= 2.0]
    if len(far_df) > 0:
        far_total = far_df['Total_Days'].sum()
        far_retraced = far_df['Retraced_Days'].sum()
        far_rate = (far_retraced / far_total * 100) if far_total >= 20 else None
    else:
        far_rate = None
    
    if close_rate is not None and far_rate is not None:
        impact = close_rate - far_rate
        impact_data.append({
            'Range': range_name,
            'Close_Rate': close_rate,
            'Far_Rate': far_rate,
            'Impact': impact
        })

impact_df = pd.DataFrame(impact_data).sort_values('Impact', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(14, 10))

y_pos = np.arange(len(impact_df))
colors = ['green' if x > 0 else 'red' for x in impact_df['Impact']]

bars = ax.barh(y_pos, impact_df['Impact'], color=colors, alpha=0.7)

# Add value labels
for i, (idx, row) in enumerate(impact_df.iterrows()):
    ax.text(row['Impact'], i, f"  {row['Impact']:+.1f}%  ", 
           va='center', fontweight='bold', fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(impact_df['Range'], fontsize=9)
ax.set_xlabel('Impact of Distance (Close <0.5% minus Far >2%)', fontsize=11, fontweight='bold')
ax.set_title('Which Ranges Are Most Affected by Distance?\n(Positive = Distance Hurts Retracement Odds)', 
            fontsize=13, fontweight='bold', pad=20)

ax.axvline(x=0, color='black', linewidth=1)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
filename = os.path.join(viz_folder, 'impact_chart.png')
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"  ✓ impact_chart.png")

# ============================================================
# 4. COMPARISON TABLE - Above vs Below Midpoint
# ============================================================

print("\n4. Creating Above vs Below Comparison...")

fig, ax = plt.subplots(figsize=(16, 12))
ax.axis('off')

table_data = []

# Get top 10 ranges
for range_name in range_success['Range'].head(10):
    range_df = df[df['Range'] == range_name]
    
    # Overall
    overall_total = range_df['Total_Days'].sum()
    overall_retraced = range_df['Retraced_Days'].sum()
    overall_rate = (overall_retraced / overall_total) * 100
    
    # Above midpoint
    above_total = range_df['Above_Midpoint_Total'].sum()
    above_retraced = range_df['Above_Midpoint_Retraced'].sum()
    above_rate = (above_retraced / above_total * 100) if above_total > 0 else 0
    
    # Below midpoint
    below_total = range_df['Below_Midpoint_Total'].sum()
    below_retraced = range_df['Below_Midpoint_Retraced'].sum()
    below_rate = (below_retraced / below_total * 100) if below_total > 0 else 0
    
    table_data.append([
        range_name,
        f"{overall_rate:.1f}%",
        f"{above_rate:.1f}%\n(n={above_total})",
        f"{below_rate:.1f}%\n(n={below_total})",
        f"{abs(above_rate - below_rate):.1f}%"
    ])

headers = ['Range', 'Overall\nSuccess', 'Opened ABOVE\nMidpoint', 'Opened BELOW\nMidpoint', 'Difference']

table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center',
                 colWidths=[0.30, 0.15, 0.20, 0.20, 0.15])

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.5)

# Style header
for i in range(len(headers)):
    table[(0, i)].set_facecolor('#000000')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Style rows
for i in range(1, len(table_data) + 1):
    for j in range(len(headers)):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#EEEEEE')

plt.title('Does It Matter If 9:30am Opens Above or Below the Range Midpoint?\n(Top 10 Ranges)',
          fontsize=13, fontweight='bold', pad=20)

filename = os.path.join(viz_folder, 'above_vs_below_comparison.png')
plt.tight_layout()
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"  ✓ above_vs_below_comparison.png")

# ============================================================
# 5. SUMMARY TABLE - Best Trading Setups
# ============================================================

print("\n5. Creating Best Setups Summary...")

fig, ax = plt.subplots(figsize=(18, 10))
ax.axis('off')

# Find best setups (high success rate when close to midpoint)
best_setups = []

for range_name in df['Range'].unique():
    range_df = df[df['Range'] == range_name]
    
    # Get close distance performance (0-0.5%)
    close_df = range_df[range_df['Max_Distance_Pct'] <= 0.5]
    
    if len(close_df) > 0:
        close_total = close_df['Total_Days'].sum()
        close_retraced = close_df['Retraced_Days'].sum()
        
        if close_total >= 30:  # Minimum sample size
            close_rate = (close_retraced / close_total) * 100
            
            # Overall rate
            overall_total = range_df['Total_Days'].sum()
            overall_retraced = range_df['Retraced_Days'].sum()
            overall_rate = (overall_retraced / overall_total) * 100
            
            best_setups.append({
                'Range': range_name,
                'Close_Rate': close_rate,
                'Close_Sample': close_total,
                'Overall_Rate': overall_rate,
                'Overall_Sample': overall_total
            })

best_df = pd.DataFrame(best_setups).sort_values('Close_Rate', ascending=False).head(15)

table_data = []
for _, row in best_df.iterrows():
    stars = "★★★" if row['Close_Rate'] >= 75 else "★★" if row['Close_Rate'] >= 70 else "★"
    
    table_data.append([
        row['Range'],
        f"{row['Close_Rate']:.1f}%",
        f"{row['Close_Sample']}",
        stars,
        f"{row['Overall_Rate']:.1f}%",
        f"{row['Overall_Sample']}"
    ])

headers = ['Range', 'Success Rate\n(Close <0.5%)', 'Close\nSample', 'Rating', 
           'Overall\nSuccess', 'Overall\nSample']

table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center',
                 colWidths=[0.30, 0.15, 0.10, 0.10, 0.15, 0.10])

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.2)

# Style header
for i in range(len(headers)):
    table[(0, i)].set_facecolor('#000000')
    table[(0, i)].set_text_props(weight='bold', color='white', size=8)

# Style rows
for i in range(1, len(table_data) + 1):
    for j in range(len(headers)):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#EEEEEE')

plt.title('BEST TRADING SETUPS: High Success When Close to Midpoint (<0.5%)\n' +
          'Trade these ranges when 9:30am opens near the 50% level',
          fontsize=12, fontweight='bold', pad=20)

filename = os.path.join(viz_folder, 'best_setups.png')
plt.tight_layout()
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"  ✓ best_setups.png")

print("\n" + "=" * 120)
print(f"✓ DONE! All distance analysis visualizations saved to:")
print(f"  {viz_folder}")
print("\nFiles created:")
print("  • 10 Distance curve charts (individual ranges)")
print("  • 1 Heatmap (all ranges vs distance)")
print("  • 1 Impact chart (close vs far comparison)")
print("  • 1 Above/Below comparison table")
print("  • 1 Best setups summary")
print("\nTotal: ~14 visualization files")
print("=" * 120)


CREATING DISTANCE vs RETRACEMENT VISUALIZATIONS

Saving visualizations to: C:\Users\tammm\NQ and ES analysis\distance_analysis_visuals
✓ Loaded 155 records

1. Creating Distance Curves for Top 10 Ranges...
  ✓ curve_PreMarket_9am_930am.png
  ✓ curve_PreMarket_8am_930am.png
  ✓ curve_PreMarket_8am_9am.png
  ✓ curve_PreMarket_7am_8am.png
  ✓ curve_PreMarket_6am_7am.png
  ✓ curve_London_5am_6am.png
  ✓ curve_London_4am_5am.png
  ✓ curve_London_3am_4am.png
  ✓ curve_London_2am_3am.png
  ✓ curve_Late_Night_1am_2am.png

2. Creating Distance Heatmap...
  ✓ heatmap_all_ranges.png

3. Creating Distance Impact Chart...
  ✓ impact_chart.png

4. Creating Above vs Below Comparison...
  ✓ above_vs_below_comparison.png

5. Creating Best Setups Summary...
  ✓ best_setups.png

✓ DONE! All distance analysis visualizations saved to:
  C:\Users\tammm\NQ and ES analysis\distance_analysis_visuals

Files created:
  • 10 Distance curve charts (individual ranges)
  • 1 Heatmap (all ranges vs distance)
  • 1 I